<a href="https://colab.research.google.com/github/Sharafatnoa/DailyPractice/blob/main/2026-10-08-statistics.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# Daily Practice — 2026-10-08: Statistics — Did the Data Shift Under Your Model's Feet?

**Dataset:** [California Housing](https://scikit-learn.org/stable/datasets/real_world.html#california-housing-dataset) — 20,640 real 1990-census block-group records with 8 numeric features, bundled with scikit-learn (`fetch_california_housing`). This is genuine census data, not a synthetic toy set.

**The scenario:** Imagine a home-valuation model was trained on listings from Southern California and is now being scored against incoming listings from further north in the state. Nobody flipped a "drift" switch for you — the only way to know the input distribution has moved is to *test* it, feature by feature, the same way a QA engineer treats a changed environment as a hypothesis to verify rather than a vibe to trust. Eyeballing an overlaid histogram is a start, but it doesn't scale to dozens of features and it doesn't tell you when a visible-looking difference is actually just noise from a modest sample size.

You'll build a small, reusable **drift-detection toolkit**: a two-sample statistical test per feature, an effect-size measure so "statistically different" doesn't get confused with "practically different," a multi-feature report with a multiple-testing correction (you're running one test per feature, and that adds up fast), and a decision rule that turns the report into one of three monitoring actions.

**What you should produce:**
1. `ks_drift_test(reference, current, alpha=0.05)` — a two-sample Kolmogorov–Smirnov test for one feature, returning the KS statistic, the p-value, and a boolean `is_drifted` flag at the given `alpha`.
2. `standardized_mean_diff(reference, current)` — Cohen's *d* effect size between the two samples' means, so you can tell a tiny-but-"significant" shift apart from a large one.
3. `generate_drift_report(reference_df, current_df, feature_cols, alpha=0.05)` — runs `ks_drift_test` and `standardized_mean_diff` on every feature in `feature_cols`, applies a **Bonferroni correction** to `alpha` across the features tested, and returns a tidy `pandas.DataFrame` with one row per feature (statistic, raw p-value, corrected threshold, effect size, `is_drifted`).
4. `summarize_alert(report_df, effect_size_threshold=0.2)` — combines the corrected statistical signal with the effect-size signal across the whole report to return one of `"no drift"`, `"monitor"`, or `"retrain alert"`.

Fill in every function marked `TODO` / `raise NotImplementedError` yourself before you look at the solution below.


## Setup

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats

from sklearn.datasets import fetch_california_housing

RNG = np.random.default_rng(42)


In [ ]:
data = fetch_california_housing(as_frame=True)
df = data.frame
print("shape:", df.shape)
df.head()


## Splitting into "reference" and "current" batches

There's no drift label in this dataset, so we create the split using a genuine, real-world axis already present in the data: **latitude**. Block groups south of latitude 36 sit in the Southern California / LA basin region; block groups at or above it sit further north (Central Coast / Bay Area and beyond). Treat the southern half as the data the model was *trained* on (the reference batch) and the northern half as a new *incoming* batch you now have to monitor — a realistic stand-in for "the model got deployed somewhere its training data didn't fully represent."


In [ ]:
reference_df = df[df["Latitude"] < 36].reset_index(drop=True)
current_df = df[df["Latitude"] >= 36].reset_index(drop=True)

print("reference rows:", len(reference_df))
print("current rows:", len(current_df))

# Features we'll monitor for drift. Latitude/Longitude are excluded since the
# split itself is defined by latitude -- testing them would be circular.
FEATURE_COLS = ["MedInc", "HouseAge", "AveRooms", "AveBedrms", "Population", "AveOccup"]


## Your turn: implement the four functions below

In [ ]:
def ks_drift_test(reference, current, alpha=0.05):
    """
    TODO: Run a two-sample Kolmogorov-Smirnov test comparing the two 1-D
    arrays `reference` and `current` (use scipy.stats.ks_2samp).

    Return a dict with keys: "statistic", "p_value", "is_drifted", where
    is_drifted is True iff p_value < alpha.
    """
    raise NotImplementedError


def standardized_mean_diff(reference, current):
    """
    TODO: Compute Cohen's d between `reference` and `current`:

        d = (mean(current) - mean(reference)) / pooled_std

    where pooled_std is the pooled standard deviation of the two samples:

        pooled_std = sqrt(((n1 - 1) * var1 + (n2 - 1) * var2) / (n1 + n2 - 2))

    Use ddof=1 (sample variance) for var1 and var2.
    """
    raise NotImplementedError


def generate_drift_report(reference_df, current_df, feature_cols, alpha=0.05):
    """
    TODO: For each feature in feature_cols:
      - run ks_drift_test on reference_df[feature] vs current_df[feature]
      - compute standardized_mean_diff for the same pair
      - apply a Bonferroni-corrected alpha: alpha / len(feature_cols)
      - recompute is_drifted against the CORRECTED alpha, not the raw one

    Return a pandas.DataFrame indexed by feature with columns:
    ["ks_statistic", "p_value", "corrected_alpha", "effect_size", "is_drifted"]
    """
    raise NotImplementedError


def summarize_alert(report_df, effect_size_threshold=0.2):
    """
    TODO: Look across every row of report_df and return one of:
      - "retrain alert"  if ANY feature is_drifted AND has
                         abs(effect_size) >= effect_size_threshold
      - "monitor"        if ANY feature is_drifted but none clears the
                         effect-size threshold
      - "no drift"       if no feature is_drifted at all

    This mirrors a real monitoring policy: a statistically detectable shift
    that's also practically large enough to matter is worth paging someone
    about; a statistically detectable but tiny shift is worth watching, not
    an immediate retrain.
    """
    raise NotImplementedError


In [ ]:
# TODO: put it together.
# 1. Call generate_drift_report on reference_df / current_df over FEATURE_COLS
#    and display the resulting DataFrame.
# 2. Call summarize_alert on that report and print the result.
# 3. Pick the single feature with the largest |effect_size| and print a
#    one-sentence, reviewer-readable explanation of why it drifted (hint:
#    look at what each feature actually measures and how that could differ
#    between Southern and Northern California block groups).


---

## Solution

*(scroll down when you're ready — try it yourself first)*

<details>
<summary>Click to reveal solution</summary>

### Implementation

```python
def ks_drift_test(reference, current, alpha=0.05):
    statistic, p_value = stats.ks_2samp(reference, current)
    return {"statistic": statistic, "p_value": p_value, "is_drifted": p_value < alpha}


def standardized_mean_diff(reference, current):
    reference = np.asarray(reference)
    current = np.asarray(current)
    n1, n2 = len(reference), len(current)
    var1, var2 = reference.var(ddof=1), current.var(ddof=1)
    pooled_std = np.sqrt(((n1 - 1) * var1 + (n2 - 1) * var2) / (n1 + n2 - 2))
    return (current.mean() - reference.mean()) / pooled_std


def generate_drift_report(reference_df, current_df, feature_cols, alpha=0.05):
    corrected_alpha = alpha / len(feature_cols)
    rows = []
    for feature in feature_cols:
        ref_vals = reference_df[feature].to_numpy()
        cur_vals = current_df[feature].to_numpy()
        ks_result = ks_drift_test(ref_vals, cur_vals, alpha=corrected_alpha)
        effect = standardized_mean_diff(ref_vals, cur_vals)
        rows.append({
            "feature": feature,
            "ks_statistic": ks_result["statistic"],
            "p_value": ks_result["p_value"],
            "corrected_alpha": corrected_alpha,
            "effect_size": effect,
            "is_drifted": ks_result["is_drifted"],
        })
    return pd.DataFrame(rows).set_index("feature")


def summarize_alert(report_df, effect_size_threshold=0.2):
    drifted = report_df[report_df["is_drifted"]]
    if drifted.empty:
        return "no drift"
    if (drifted["effect_size"].abs() >= effect_size_threshold).any():
        return "retrain alert"
    return "monitor"
```

### Putting it together

```python
report = generate_drift_report(reference_df, current_df, FEATURE_COLS)
print(report.round(4))

print("\nalert level:", summarize_alert(report))

biggest = report["effect_size"].abs().idxmax()
print(f"\nlargest shift: {biggest} (effect size = {report.loc[biggest, 'effect_size']:.3f})")
```

With this real latitude-based split, the report typically shows **every** feature as `is_drifted = True` even after the Bonferroni correction (20,000+ rows per batch gives the KS test enormous power to detect even small distributional differences), but the `effect_size` column separates the signal that matters from the noise. `MedInc` (median income) and `HouseAge` tend to show the largest effect sizes — Bay Area block groups in this 1990 data have both notably higher incomes and, in many tracts, different housing-stock age profiles than Southern California — while `AveBedrms` often shows a `is_drifted = True` with a small effect size, i.e. "technically different, not practically meaningful."

**The point:** with a large enough sample, a p-value alone will flag almost anything as "significant." That's exactly why a drift-monitoring pipeline that pages someone on `p < 0.05` alone, without an effect-size gate, will drown its on-call engineer in alerts for shifts too small to affect predictions. The same discipline that keeps a QA suite from crying wolf on noisy flaky assertions applies here: pair the hypothesis test with a practical-significance threshold before it triggers an action.

### Bonus: an alternative to KS — Population Stability Index (PSI)

KS is sensitive but doesn't by itself give you a single severity number teams can threshold the way they threshold PSI in traditional MLOps/credit-risk monitoring. A quick PSI implementation for comparison:

```python
def psi(reference, current, bins=10):
    reference = np.asarray(reference)
    current = np.asarray(current)
    quantile_edges = np.quantile(reference, np.linspace(0, 1, bins + 1))
    quantile_edges[0], quantile_edges[-1] = -np.inf, np.inf
    ref_counts, _ = np.histogram(reference, bins=quantile_edges)
    cur_counts, _ = np.histogram(current, bins=quantile_edges)
    ref_pct = np.clip(ref_counts / len(reference), 1e-6, None)
    cur_pct = np.clip(cur_counts / len(current), 1e-6, None)
    return float(np.sum((cur_pct - ref_pct) * np.log(cur_pct / ref_pct)))

for feature in FEATURE_COLS:
    print(feature, round(psi(reference_df[feature], current_df[feature]), 3))
```

A common rule of thumb in production monitoring: PSI < 0.1 is no real shift, 0.1-0.25 is moderate (worth watching), and > 0.25 is a major shift worth acting on — the same three-tier logic `summarize_alert` implements above, just calibrated on a different statistic.

</details>